In [1]:
from pathlib import Path

def resolve_project_root() -> Path:
    candidates = [Path.cwd(), Path.cwd().parent]
    for candidate in candidates:
        if (candidate / "text_classification").exists():
            return candidate.resolve()
    return Path.cwd().resolve()

PROJECT_ROOT = resolve_project_root()
NOTEBOOK_DIR = PROJECT_ROOT / "text_classification"
ARTIFACT_DIR = NOTEBOOK_DIR / "artifacts"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"ARTIFACT_DIR = {ARTIFACT_DIR}")



PROJECT_ROOT = D:\hvc
ARTIFACT_DIR = D:\hvc\text_classification\artifacts


In [2]:
from pathlib import Path
CSV_FILE = Path('datasets') /'text-dataset'/'ALL_TEXTS_LABELED.csv'


In [3]:
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report
import numpy as np

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "roberta-base"

# -----------------------
# 1. LOAD + BALANCE DATA
# -----------------------
df = pd.read_csv(CSV_FILE)

# df = pd.read_csv("dataset.csv")

# remove NaNs
df = df.dropna(subset=["text"])

# convert everything to string
df["text"] = df["text"].astype(str)

# remove empty strings
df = df[df["text"].str.strip() != ""]

# OPTIONAL: remove very short garbage
df = df[df["text"].str.len() > 3]

LABEL_MAP = {
    "label_0": 0,
    "label_1": 1,
    "label_2": 2,
    "label_3": 3
}

df["severity"] = df["label"].map(LABEL_MAP)

# Binary mapping
df["binary"] = df["severity"].apply(lambda x: 0 if x == 0 else 1)

# Stratified sampling (equal distribution)
samples_per_class = 20000 

# balanced_df = df.groupby("severity", group_keys=False).apply(lambda x: x.sample(samples_per_class, random_state=42)).reset_index(drop=True)

# samples_per_class = 1250

balanced_df = pd.concat([
    df[df["severity"] == cls].sample(
        samples_per_class,
        replace=False,  # all classes have enough data
        random_state=42
    )
    for cls in [0, 1, 2, 3]
]).sample(frac=1, random_state=42).reset_index(drop=True)

C:\Users\ASUS\AppData\Local\Temp\ipykernel_37264\2983123820.py:16: DtypeWarning: Columns (0: text) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(CSV_FILE)


In [4]:
print(df["severity"].value_counts())

severity
0    2677124
2    2161585
1      81754
3      42961
Name: count, dtype: int64


In [5]:
df

,text,label,severity,binary
0,He got his money... now he lies in wait till a...,label_2,2,1
1,Mad dog will surely put the liberals in mental...,label_2,2,1
2,And Trump continues his lifelong cowardice by ...,label_2,2,1
3,"""while arresting a man for resisting arrest"".\...",label_2,2,1
4,Tucker and Paul are both total bad ass mofo's.,label_2,2,1
...,...,...,...,...
5071667,"**oh wow, and on what website was this receip...",label_0,0,0
5071668,== 12th Doctor == LOOK! I HAVE HAD ENOUGH Y...,label_2,2,1
5071669,==Date confusion == If he was born in June ...,label_0,0,0
5071670,"Hi, Since my post of above,I see that you h...",label_0,0,0


In [6]:
balanced_df

,text,label,severity,binary
0,Prestanite nam mazati nos svojim seksualnim or...,label_2,2,1
1,pointless in that charlie died to allow the ca...,label_2,2,1
2,i absolutely must love no the plan but to kill...,label_3,3,1
3,bullets through a nigguh true jeans,label_3,3,1
4,undocumented migrants are going to destroy ame...,label_2,2,1
...,...,...,...,...
79995,He's leaving messages on the image file's talk...,label_0,0,0
79996,flowers is a big old queen fuck off a gay boi,label_2,2,1
79997,Tentioner THANK YOU VERY MUCH. I APRECIATE THE...,label_3,3,1
79998,well you certainly sound sane here again not r...,label_0,0,0


In [7]:
print(balanced_df["severity"].value_counts())


severity
2    20000
3    20000
0    20000
1    20000
Name: count, dtype: int64


In [8]:
import logging
import time

# -----------------------
# LOGGING SETUP
# -----------------------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s"
)

# -----------------------
# 2. SPLIT
# -----------------------
logging.info("Starting train/val split")

train_df, val_df = train_test_split(
    balanced_df,
    test_size=0.2,
    stratify=balanced_df["severity"],
    random_state=42
)

logging.info(f"Train size: {len(train_df)} | Val size: {len(val_df)}")
logging.info(f"Train distribution:\n{train_df['severity'].value_counts()}")

# -----------------------
# 3. DATASET
# -----------------------
logging.info("Loading tokenizer")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class TextDataset(Dataset):
    def __init__(self, df, max_len=128):
        self.texts = df["text"].astype(str).tolist()
        self.binary = df["binary"].tolist()
        self.severity = df["severity"].tolist()
        self.max_len = max_len

        logging.info(f"Dataset initialized with {len(self.texts)} samples")

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])

        enc = tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )

        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "binary": torch.tensor(self.binary[idx], dtype=torch.float),
            "severity": torch.tensor(self.severity[idx], dtype=torch.long)
        }

logging.info("Creating datasets")

train_dataset = TextDataset(train_df)
val_dataset = TextDataset(val_df)

logging.info("Creating dataloaders")

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=128)

# -----------------------
# 4. MODEL
# -----------------------
logging.info("Loading model")

class MultiTaskModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.encoder = AutoModel.from_pretrained(MODEL_NAME)
        hidden = self.encoder.config.hidden_size

        # Enable gradient checkpointing (saves memory)
        if hasattr(self.encoder, "gradient_checkpointing_enable"):
            self.encoder.gradient_checkpointing_enable()

        # -------- Feature extractor --------
        self.feature_layer = nn.Sequential(
            nn.Linear(hidden * 2, hidden),
            nn.LayerNorm(hidden),
            nn.ReLU(),
            nn.Dropout(0.3)
        )

        # -------- Binary head --------
        self.binary_head = nn.Sequential(
            nn.Linear(hidden, hidden // 2),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden // 2, 1)
        )

        # -------- Severity head --------
        self.severity_head = nn.Sequential(
            nn.Linear(hidden, hidden // 2),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden // 2, 4)
        )

    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)

        # CLS token
        cls = out.last_hidden_state[:, 0]

        # Mean pooling (important improvement)
        mean_pool = (out.last_hidden_state * attention_mask.unsqueeze(-1)).sum(1)
        mean_pool = mean_pool / attention_mask.sum(1, keepdim=True)

        # Combine CLS + mean (richer representation)
        combined = torch.cat([cls, mean_pool], dim=1)

        features = self.feature_layer(combined)

        binary = self.binary_head(features).squeeze(-1)
        severity = self.severity_head(features)

        return binary, severity

model = MultiTaskModel().to(DEVICE)

logging.info(f"Model loaded on {DEVICE}")

# -----------------------
# 5. LOSS + OPTIMIZER
# -----------------------
logging.info("Initializing optimizer and loss")

bce_loss = nn.BCEWithLogitsLoss()
ce_loss = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

# -----------------------
# 6. TRAIN
# -----------------------
def train_epoch(epoch):
    model.train()
    total_loss = 0
    start_time = time.time()

    logging.info(f"Epoch {epoch} training started")

    for step, batch in enumerate(train_loader):
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        binary = batch["binary"].to(DEVICE)
        severity = batch["severity"].to(DEVICE)

        optimizer.zero_grad()

        binary_logits, severity_logits = model(input_ids, attention_mask)

        loss = (
            bce_loss(binary_logits, binary) +
            0.7 * ce_loss(severity_logits, severity)
        )

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        # Log every 50 steps
        if step % 50 == 0:
            logging.info(f"[Epoch {epoch} | Step {step}] Loss: {loss.item():.4f}")

    avg_loss = total_loss / len(train_loader)
    elapsed = time.time() - start_time

    logging.info(f"Epoch {epoch} completed | Loss: {avg_loss:.4f} | Time: {elapsed:.2f}s")

    return avg_loss

# -----------------------
# 7. EVAL
# -----------------------
def evaluate(epoch):
    model.eval()

    total_loss = 0
    total_samples = 0

    all_bin_preds = []
    all_bin_labels = []

    all_sev_preds = []
    all_sev_labels = []

    logging.info(f"Epoch {epoch} evaluation started")

    start_time = time.time()

    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)

            binary_labels = batch["binary"].to(DEVICE)
            severity_labels = batch["severity"].to(DEVICE)

            binary_logits, severity_logits = model(input_ids, attention_mask)

            # -------- LOSS --------
            loss = (
                bce_loss(binary_logits, binary_labels) +
                0.7 * ce_loss(severity_logits, severity_labels)
            )

            total_loss += loss.item() * input_ids.size(0)
            total_samples += input_ids.size(0)

            # -------- PREDICTIONS --------
            bin_probs = torch.sigmoid(binary_logits)
            bin_preds = (bin_probs > 0.5).long()

            sev_preds = torch.argmax(severity_logits, dim=1)

            all_bin_preds.extend(bin_preds.cpu().numpy())
            all_bin_labels.extend(binary_labels.cpu().numpy())

            all_sev_preds.extend(sev_preds.cpu().numpy())
            all_sev_labels.extend(severity_labels.cpu().numpy())

    # -------- METRICS --------
    val_loss = total_loss / total_samples

    # Binary accuracy
    bin_acc = np.mean(np.array(all_bin_preds) == np.array(all_bin_labels))

    # Severity accuracy
    sev_acc = np.mean(np.array(all_sev_preds) == np.array(all_sev_labels))

    # F1 scores
    bin_f1 = f1_score(all_bin_labels, all_bin_preds)
    sev_f1 = f1_score(all_sev_labels, all_sev_preds, average="macro")

    # -------- LOGS --------
    logging.info(f"Epoch {epoch} Validation Loss: {val_loss:.4f}")
    logging.info(f"Binary Accuracy: {bin_acc:.4f} | F1: {bin_f1:.4f}")
    logging.info(f"Severity Accuracy: {sev_acc:.4f} | F1: {sev_f1:.4f}")

    logging.info("\nSeverity Classification Report:\n" +
                 classification_report(all_sev_labels, all_sev_preds))

    elapsed = time.time() - start_time
    logging.info(f"Evaluation time: {elapsed:.2f}s")

    return val_loss, bin_acc, sev_acc, sev_f1

2026-04-15 15:16:58,659 | INFO | Starting train/val split
2026-04-15 15:16:58,683 | INFO | Train size: 64000 | Val size: 16000
2026-04-15 15:16:58,684 | INFO | Train distribution:
severity
1    16000
2    16000
3    16000
0    16000
Name: count, dtype: int64
2026-04-15 15:16:58,685 | INFO | Loading tokenizer
2026-04-15 15:16:59,023 | INFO | HTTP Request: HEAD https://huggingface.co/roberta-base/resolve/main/config.json "HTTP/1.1 200 OK"
2026-04-15 15:16:59,318 | INFO | HTTP Request: HEAD https://huggingface.co/roberta-base/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"
2026-04-15 15:16:59,589 | INFO | HTTP Request: GET https://huggingface.co/api/models/roberta-base/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 307 Temporary Redirect"
2026-04-15 15:16:59,868 | INFO | HTTP Request: GET https://huggingface.co/api/models/FacebookAI/roberta-base/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-04-15 15:17:00,12

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
2026-04-15 15:17:02,247 | INFO | Model loaded on cuda
2026-04-15 15:17:02,248 | INFO | Initializing optimizer and loss


In [9]:

# -----------------------
# 8. RUN
# -----------------------
for epoch in range(10):
    loss = train_epoch(epoch)
    val_loss, bin_acc, sev_acc, sev_f1 = evaluate(epoch)

    print(
        f"Epoch {epoch} | "
        f"Train Loss: {loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Bin Acc: {bin_acc:.4f} | "
        f"Sev Acc: {sev_acc:.4f} | "
        f"Sev F1: {sev_f1:.4f}"
    ) 
    # print(f"Epoch {epoch} | Loss: {loss:.4f} | F1: {f1:.4f}")

2026-04-15 15:17:02,264 | INFO | Epoch 0 training started
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
2026-04-15 15:17:04,979 | INFO | [Epoch 0 | Step 0] Loss: 1.6604
2026-04-15 15:19:03,537 | INFO | [Epoch 0 | Step 50] Loss: 0.8017
2026-04-15 15:21:02,273 | INFO | [Epoch 0 | Step 100] Loss: 0.5530
2026-04-15 15:23:01,136 | INFO | [Epoch 0 | Step 150] Loss: 0.4470
2026-04-15 15:25:00,287 | INFO | [Epoch 0 | Step 200] Loss: 0.4762
2026-04-15 15:26:57,568 | INFO | Epoch 0 completed | Loss: 0.6130 | Time: 595.30s
2026-04-15 15:26:57,571 | INFO | Epoch 0 evaluation started
2026-04-15 15:27:42,379 | INFO | Epoch 0 Validation Loss: 0.3779
2026-04-15 15:27:42,380 | INFO | Binary Accuracy: 0.9344 | F1: 0.9560
2026-04-15 15:27:42,381 | INFO | Severity Accuracy: 0.8911 | F1: 0.8903
2026-04-15 15:27:42,391 | INFO | 
Severity Classification Report:
              precision    recall  f1-score   support

           0       0.85      0.89      0.87      40

Epoch 0 | Train Loss: 0.6130 | Val Loss: 0.3779 | Bin Acc: 0.9344 | Sev Acc: 0.8911 | Sev F1: 0.8903


2026-04-15 15:27:44,892 | INFO | [Epoch 1 | Step 0] Loss: 0.3286
2026-04-15 15:29:44,645 | INFO | [Epoch 1 | Step 50] Loss: 0.3150
2026-04-15 15:34:50,106 | INFO | [Epoch 1 | Step 100] Loss: 0.2755
2026-04-15 15:36:49,867 | INFO | [Epoch 1 | Step 150] Loss: 0.3519
2026-04-15 15:38:49,520 | INFO | [Epoch 1 | Step 200] Loss: 0.2876
2026-04-15 15:40:47,023 | INFO | Epoch 1 completed | Loss: 0.3412 | Time: 784.63s
2026-04-15 15:40:47,026 | INFO | Epoch 1 evaluation started
2026-04-15 15:41:55,910 | INFO | Epoch 1 Validation Loss: 0.3618
2026-04-15 15:41:55,911 | INFO | Binary Accuracy: 0.9375 | F1: 0.9582
2026-04-15 15:41:55,912 | INFO | Severity Accuracy: 0.8956 | F1: 0.8944
2026-04-15 15:41:55,922 | INFO | 
Severity Classification Report:
              precision    recall  f1-score   support

           0       0.86      0.89      0.88      4000
           1       0.91      0.95      0.93      4000
           2       0.89      0.79      0.83      4000
           3       0.92      0.95   

Epoch 1 | Train Loss: 0.3412 | Val Loss: 0.3618 | Bin Acc: 0.9375 | Sev Acc: 0.8956 | Sev F1: 0.8944


2026-04-15 15:41:58,363 | INFO | [Epoch 2 | Step 0] Loss: 0.3286
2026-04-15 15:43:57,519 | INFO | [Epoch 2 | Step 50] Loss: 0.2762
2026-04-15 15:45:57,666 | INFO | [Epoch 2 | Step 100] Loss: 0.2630
2026-04-15 15:47:58,966 | INFO | [Epoch 2 | Step 150] Loss: 0.2491
2026-04-15 15:49:59,036 | INFO | [Epoch 2 | Step 200] Loss: 0.2720
2026-04-15 15:51:57,660 | INFO | Epoch 2 completed | Loss: 0.2631 | Time: 601.73s
2026-04-15 15:51:57,663 | INFO | Epoch 2 evaluation started
2026-04-15 15:52:41,659 | INFO | Epoch 2 Validation Loss: 0.3474
2026-04-15 15:52:41,660 | INFO | Binary Accuracy: 0.9404 | F1: 0.9604
2026-04-15 15:52:41,661 | INFO | Severity Accuracy: 0.9039 | F1: 0.9032
2026-04-15 15:52:41,676 | INFO | 
Severity Classification Report:
              precision    recall  f1-score   support

           0       0.89      0.87      0.88      4000
           1       0.93      0.95      0.94      4000
           2       0.87      0.82      0.85      4000
           3       0.93      0.96   

...

Epoch 7 | Train Loss: 0.0825 | Val Loss: 0.4662 | Bin Acc: 0.9397 | Sev Acc: 0.9051 | Sev F1: 0.9051


2026-04-15 16:54:22,434 | INFO | [Epoch 8 | Step 0] Loss: 0.0327
2026-04-15 16:57:48,002 | INFO | [Epoch 8 | Step 50] Loss: 0.0522
2026-04-15 16:59:48,802 | INFO | [Epoch 8 | Step 100] Loss: 0.0722
2026-04-15 17:01:48,551 | INFO | [Epoch 8 | Step 150] Loss: 0.0457
2026-04-15 17:03:48,435 | INFO | [Epoch 8 | Step 200] Loss: 0.0212
2026-04-15 17:05:45,312 | INFO | Epoch 8 completed | Loss: 0.0679 | Time: 685.40s
2026-04-15 17:05:45,314 | INFO | Epoch 8 evaluation started
2026-04-15 17:08:06,901 | INFO | Epoch 8 Validation Loss: 0.5075
2026-04-15 17:08:06,901 | INFO | Binary Accuracy: 0.9413 | F1: 0.9607
2026-04-15 17:08:06,902 | INFO | Severity Accuracy: 0.9059 | F1: 0.9048
2026-04-15 17:08:06,912 | INFO | 
Severity Classification Report:
              precision    recall  f1-score   support

           0       0.88      0.89      0.88      4000
           1       0.93      0.96      0.95      4000
           2       0.89      0.80      0.84      4000
           3       0.93      0.97   

Epoch 8 | Train Loss: 0.0679 | Val Loss: 0.5075 | Bin Acc: 0.9413 | Sev Acc: 0.9059 | Sev F1: 0.9048


2026-04-15 17:08:09,324 | INFO | [Epoch 9 | Step 0] Loss: 0.0857
2026-04-15 17:10:09,425 | INFO | [Epoch 9 | Step 50] Loss: 0.0305
2026-04-15 17:12:08,680 | INFO | [Epoch 9 | Step 100] Loss: 0.0444
2026-04-15 17:14:26,546 | INFO | [Epoch 9 | Step 150] Loss: 0.0946
2026-04-15 17:16:25,743 | INFO | [Epoch 9 | Step 200] Loss: 0.0208
2026-04-15 17:18:23,007 | INFO | Epoch 9 completed | Loss: 0.0598 | Time: 616.09s
2026-04-15 17:18:23,011 | INFO | Epoch 9 evaluation started
2026-04-15 17:19:07,668 | INFO | Epoch 9 Validation Loss: 0.5125
2026-04-15 17:19:07,669 | INFO | Binary Accuracy: 0.9411 | F1: 0.9609
2026-04-15 17:19:07,669 | INFO | Severity Accuracy: 0.9079 | F1: 0.9078
2026-04-15 17:19:07,679 | INFO | 
Severity Classification Report:
              precision    recall  f1-score   support

           0       0.89      0.87      0.88      4000
           1       0.95      0.95      0.95      4000
           2       0.85      0.85      0.85      4000
           3       0.94      0.96   

Epoch 9 | Train Loss: 0.0598 | Val Loss: 0.5125 | Bin Acc: 0.9411 | Sev Acc: 0.9079 | Sev F1: 0.9078


In [11]:
torch.save({
    "model_state_dict": model.state_dict(),
    "model_name": MODEL_NAME
}, "roberta_base_finetuned_dualhead.pt")

In [ ]:
checkpoint = torch.load("roberta_base_finetuned_dualhead.pt")

MODEL_NAME = checkpoint["model_name"]

model = MultiTaskModel()  # uses MODEL_NAME internally
model.load_state_dict(checkpoint["model_state_dict"])